E3 Intermediate

Brian Bannon

In chemical process engineering, reactor design and operation play a critical role in achieving desired product yield, selectivity, and energy efficiency. Many industrial processes use multiple reactors in series or parallel to optimize conversion while managing reaction kinetics, heat, and residence time. Temperature is one of the most important operating variables, as it directly affects reaction rates, energy consumption, and product quality. Understanding how conversion varies with temperature and reactor configuration allows engineers to design efficient, cost-effective chemical processes.  
This notebook presents an analysis of reactor networks and temperature optimization for hypothetical polymer production scenarios. It implements four core Python functions to calculate series conversion, temperature-dependent reaction profiles, optimal operating temperatures, and the performance of multiple reactor configurations. Three test scenarios are evaluated using numerical calculations and professional visualizations to compare single versus series reactors, assess multi-temperature operation, and determine energy savings. The results provide insights into the influence of kinetic parameters, reactor staging, and operating conditions on overall process performance.

In [248]:
import numpy as np
import matplotlib.pyplot as plt

In [249]:
R = 8.314

In [250]:
def calculate_rate_constant(temperature: float, a: float, ea: float) -> float:
    """
    Function to calculate rate constant
    :param temperature: temperature in Kelvin
    :param a: pre-exponential factor in 1/s
    :param ea: activation energy in J/mol
    :return: reaction rate constant k in 1/s
    """
    return a * np.exp(-ea / (R * temperature))

In [251]:
def calculate_series_conversion(conversions: list) -> np.float64:
    """
    Function to calculate overall conversion for reactors in series
    :param conversions: list of conversions
    :return: overall conversion for reactors in series
    """
    list2 = 1 - np.array(conversions)
    return 1 - np.prod(list2)

In [252]:
def calculate_temperature_profile(t_array: np.ndarray, a: float, ea: float, time: float) -> tuple[np.ndarray, np.ndarray]:
    """
    Function to calculate temperature profile
    :param t_array: array of temperatures
    :param a: pre-exponential factor in 1/s
    :param ea: activation energy in J/mol
    :param time: reaction time in s
    :return: tuple of rate constants and conversions
    """
    k_array = a * np.exp(-ea / (R * t_array))
    x_array = 1 - np.exp(-k_array * time)
    return k_array, x_array

In [253]:
def find_optimal_temperature(t_min: float, t_max: float, a: float, ea: float, time: float, target_conversion: float) -> dict:
    """
    Function to find optimal temperature
    :param t_min: minimum temperature
    :param t_max: maximum temperature
    :param a: pre-exponential factor in 1/s
    :param ea: activation energy in J/mol
    :param time: reaction time in s
    :param target_conversion: desired conversion from 0 to 1
    :return: dictionary with 4 keys
    """
    temps = np.linspace(t_min, t_max, 1000)
    k_array, x_array = calculate_temperature_profile(temps, a, ea, time)

    indices = np.where(x_array >= target_conversion)

    if len(indices[0]) > 0:
        i = indices[0][0]
        return {
            'optimal_T': temps[i],
            'optimal_k': k_array[i],
            'optimal_X': x_array[i],
            'margin': x_array[i] - target_conversion
        }

In [254]:
def compare_reactor_configurations(config_specs: dict) -> dict[str, dict]:
    """
    :param config_specs: dict with configuration specs
    :return: dict with 3 keys
    """
    a = config_specs['A']
    ea = config_specs['Ea']
    temp = config_specs['temperature']
    n_series = config_specs['n_series']
    total_available_time = config_specs['total_available_time']
    target_conversion = config_specs['target_conversion']

    k = calculate_rate_constant(temp, a, ea)

    names = 'single_reactor', 'two_series', 'three_series'

    output = {
        names[0]: {},
        names[1]: {},
        names[2]: {}
    }

    for i in range(1, n_series + 1):
        name = names[i-1]
        tau = total_available_time / i
        conversion = k * tau / (1 + k * tau)
        conversion_overall = calculate_series_conversion([conversion] * i)

        output[name]['individual conversion'] = conversion
        output[name]['overall conversion'] = conversion_overall
        output[name]['time_per_reactor'] = tau
        output[name]['meets_target'] = conversion_overall >= target_conversion

    return output

In [ ]:
# Scenario 1
a = 3.5e6
ea = 68000
batch_time = 5400
target = 0.9
t_min = 320
t_max = 380
t_current = 360

temps = np.linspace(t_min, t_max, 100)

k_array, x_array = calculate_temperature_profile(temps, a, ea, batch_time)

plt.figure(figsize=(8,6))
plt.plot(temps, x_array * 100, label='Conversion vs Temperature')
plt.axhline(y=target*100, linestyle='--', label=f'Target ({target*100}%)')

optimal = find_optimal_temperature(t_min, t_max, a, ea, batch_time, target)

t_optimal = optimal['optimal_T']
x_optimal = optimal['optimal_X'] * 100

plt.scatter(t_optimal, x_optimal, label=f'Optimal T = {t_optimal:.1f} K')

k_current = a * np.exp(-ea / (R * t_current))
x_current = (1 - np.exp(-k_current * batch_time)) * 100

plt.scatter(t_current, x_current, label=f'Current T = {t_current} K')
plt.xlabel('Temperature (K)')
plt.ylabel('Conversion (%)')
plt.title('Polymer Production Temperature Optimization: Conversion vs Temperature')
plt.legend()
plt.grid()

energy_current = t_current - 293
energy_optimal = t_optimal - 293

energy_saved = energy_current - energy_optimal
percent_savings = energy_saved / energy_current

i1 = np.where(temps > t_optimal - 5)[0][0]
i2 = np.where(temps > t_optimal + 5)[0][0]

x_current, percent_savings*100, x_array[i1]*100, x_array[i2]*100

In [ ]:
# Scenario 2
a = 12e7
ea = 75000
temp = 350
n_series = 3
total_available_time = 7200
target_conversion = 0.95

config_specs = {
    'A': a,
    'Ea': ea,
    'temperature': temp,
    'n_series': n_series,
    'total_available_time': total_available_time,
    'target_conversion': target_conversion
}

reactors = compare_reactor_configurations(config_specs)
for i in reactors:
    print(reactors[i])
    config_names = list(reactors.keys())
overall_conversion = [reactors[name]['overall conversion']*100 for name in config_names]  # as %
individual_conversion = [reactors[name]['individual conversion']*100 for name in config_names]  # optional

plt.figure(figsize=(8,6))

bars = plt.bar(config_names, overall_conversion, color=['skyblue','orange','green'])

plt.axhline(y=target_conversion*100, color='red', linestyle='--', label=f'Target ({target_conversion*100:.0f}%)')

for bar, val in zip(bars, overall_conversion):
    plt.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', ha='center', va='bottom')

plt.xlabel('Reactor Configuration')
plt.ylabel('Overall Conversion (%)')
plt.title('Series Reactor Network Comparison')
plt.ylim(0, 105)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Scenario 3
a = 1.5e8
ea = 80000
t1 = 340
t2 = 360
t3 = 380
temps = np.array((t1, t2, t3))
time_per_reactor = 1800
target_conversion = 0.92

k_array = a * np.exp(-ea / (R * temps))
x_array = k_array * time_per_reactor / (1 + k_array * time_per_reactor)

x_cumulative = 1 - np.cumprod(1 - x_array)

plt.figure(figsize=(8,6))
plt.plot(np.arange(1,4), x_cumulative*100, marker='o', linestyle='-', label='Cumulative Conversion (%)')
plt.axhline(y=target_conversion*100, color='red', linestyle='--', label=f'Target ({target_conversion*100:.0f}%)')
plt.xlabel('Reactor Number')
plt.ylabel('Cumulative Conversion (%)')
plt.title('Multi-Temperature Series Reactor Performance (Vectorized)')
plt.xticks([1,2,3])
plt.ylim(0, 105)
plt.grid()
plt.legend()
plt.show()

k_single = calculate_rate_constant(360, a, ea)
x_single = k_single * (3*time_per_reactor) / (1 + k_single * (3*time_per_reactor))

for i, (xi, xi_cum) in enumerate(zip(x_array, x_cumulative), 1):
    print(f"Reactor {i}: Individual conversion = {xi*100:.2f}%, Cumulative conversion = {xi_cum*100:.2f}%")

print(f"Single reactor at 360 K for 5400 s: Conversion = {x_single*100:.2f}%")

QUESTION 1  
The energy savings is about 2.5%.

QUESTION 2  
Multiple smaller reactors in series achieve higher conversion because splitting the total time into smaller stages increases the individual conversions and multiplying the fractions results in a smaller product, giving a higher overal conversion.

QUESTION 3  
The second reactor contributes the most to overall conversion, adding more than the third because cumulative conversion depends on the remaining unreacted fraction; each stage converts a fraction of what remains, so even though the third reaction is hotter, there is less unreacted material left for it to act on.

QUESTION 4  
With a 5K variation, conversions are roughly between 81% and 96%. The target is sometimes achievable.

QUESTION 5  
With a lower activation energy, the reaction becomes less sensitive to temperature, so the advantage of splitting time into multiple reactors decreases, and all configurations (single, two, three in series) would have more similar conversions, reducing the performance gap between them.

The analysis demonstrates that reactor configuration and operating temperature significantly influence overall conversion and process efficiency. In Scenario 1, optimizing the reactor temperature allowed the target conversion to be met while reducing energy consumption by approximately 2.5%, highlighting the importance of temperature control. Scenario 2 showed that splitting total residence time across multiple reactors in series increases overall conversion compared to a single reactor, due to the compounding effect of staged conversions. Scenario 3 illustrated that multi-temperature operation can further improve cumulative conversion, with earlier reactor stages often contributing more to overall gain than later, hotter stages.  
Overall, these results emphasize the value of combining chemical kinetics with process optimization techniques. By carefully selecting operating temperatures and reactor arrangements, engineers can maximize product yield while minimizing energy costs. The methodologies implemented here—including vectorized calculations, temperature profiling, and configuration comparison—provide a robust framework for evaluating and optimizing real-world chemical reactor networks.